# Energy Poverty Prediction – Fixed Spatial Hold-out (PT192 & PT196)

This notebook is identical in structure to the baseline model, **except for the train/test split logic**.

**Test set definition (hard spatial hold-out):**
- All freguesias whose `ID` starts with:
  - `192` → Região de Coimbra (NUTS III: PT192)
  - `196` → Beiras e Serra da Estrela (NUTS III: PT196)

This creates a *theory-driven*, geographically coherent out-of-sample test set,
explicitly designed to stress-test spatial generalization and support sociological critique.

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet

try:
    from scipy.stats import spearmanr
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False

pd.set_option("display.max_columns", 200)

In [ ]:
# ---------- Paths ----------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.append(str(REPO_ROOT))

from utils.paths import load_paths, path_value, repo_data_path

PATHS = load_paths()

ADM_CSV = repo_data_path(PATHS, "all_used_adm_indicators.csv")
SAT_CSV = repo_data_path(PATHS, "all_used_sat_indicators.csv")
EPVI_CSV = path_value(PATHS, "epvi_csv")
SPLIT_JSON = repo_data_path(PATHS, "adm_data_split.json")

def read_csv_robust(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except Exception:
            pass
    raise RuntimeError(f"Could not read {path}")

adm = read_csv_robust(ADM_CSV)
sat = read_csv_robust(SAT_CSV)
epvi = read_csv_robust(EPVI_CSV)

with open(SPLIT_JSON, "r", encoding="utf-8") as f:
    adm_split = json.load(f)

basic_cols = adm_split["basic"]
detailed_cols = adm_split["detailed"]


In [ ]:
# ---------- Merge ----------
data = (
    epvi.drop_duplicates("ID")
    .merge(adm.drop_duplicates("ID"), on="ID", how="inner")
    .merge(sat.drop_duplicates("ID"), on="ID", how="inner")
)

TARGETS = ["EPG heating", "EPG cooling", "AIAM", "EPVI heating", "EPVI cooling"]

sat_cols = [c for c in sat.columns if c != "ID"]
basic_cols = [c for c in basic_cols if c in data.columns]
detailed_cols = [c for c in detailed_cols if c in data.columns]

X_cols = sat_cols + basic_cols

data = data.dropna(subset=TARGETS).copy()
print("Total usable rows:", len(data))

In [ ]:
# ---------- FIXED SPATIAL HOLD-OUT ----------
# Test set = freguesias whose ID starts with 192 or 196

data["ID_str"] = data["ID"].astype(str)

test_mask = data["ID_str"].str.startswith(("192", "196"))
train_df = data.loc[~test_mask].copy()
test_df = data.loc[test_mask].copy()

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Test share:", len(test_df) / len(data))

cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# ---------- Models ----------
preprocess = ColumnTransformer(
    [("num",
      Pipeline([
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
      ]),
      X_cols)],
    remainder="drop"
)

candidate_models = {
    "HGBR": HistGradientBoostingRegressor(random_state=42),
    "RF": RandomForestRegressor(
        n_estimators=600,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),
    "ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.2, max_iter=5000)
}

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def spearman(y_true, y_pred):
    if not HAVE_SCIPY:
        return np.nan
    return spearmanr(y_true, y_pred).correlation

In [ ]:
# ---------- Train & Evaluate ----------
reports = {}

for target in TARGETS:
    X_train = train_df[X_cols]
    y_train = train_df[target].astype(float)
    X_test = test_df[X_cols]
    y_test = test_df[target].astype(float)

    best_r2 = -np.inf
    best_pipe = None
    best_name = None

    for name, model in candidate_models.items():
        pipe = Pipeline([
            ("prep", preprocess),
            ("model", model)
        ])

        cv = cross_validate(
            pipe, X_train, y_train,
            cv=cv_splitter,
            scoring="r2",
            n_jobs=-1
        )

        mean_r2 = cv["test_score"].mean()
        if mean_r2 > best_r2:
            best_r2 = mean_r2
            best_pipe = pipe
            best_name = name

    best_pipe.fit(X_train, y_train)
    preds = best_pipe.predict(X_test)

    reports[target] = {
        "best_model": best_name,
        "test_r2": r2_score(y_test, preds),
        "test_mae": mean_absolute_error(y_test, preds),
        "test_rmse": rmse(y_test, preds),
        "test_spearman": spearman(y_test, preds)
    }

    print(f"\n{target}")
    for k, v in reports[target].items():
        print(f"{k}: {v}")

## Interpretation note

This split is **intentionally harsh**:

- The model never sees **central Portugal’s coastal–mountain gradient** during training.
- Any performance drop compared to random or weakly-spatial splits is *expected* and
  analytically valuable.
- Residuals here are strong evidence for:
  - spatial proxy learning,
  - missing social mechanisms,
  - limits of satellite observability.

This setup is excellent for a *methodological + sociological critique* chapter.